<!--nav--> [🗺 Learning path](README.md) · **2/48** · ◀ [Simple MultiGPU Training](./Simple_MultiGPU_Training.ipynb) · [Simple MultiGPU Benchmark](./Simple_MultiGPU_Benchmark.ipynb) ▶

# Simple Multi-GPU Full Fine-Tuning

Actually train a tiny model end-to-end on multiple GPUs. No LoRA — full parameter training.

- **Model:** GPT-2 (124M params) — small enough for full fine-tune on free GPUs
- **Dataset:** OpenWebText (subset) — real pre-training style data  
- **Method:** DeepSpeed ZeRO-3 + CPU offload — splits model across GPUs
- **Platform:** Kaggle 2x T4 (free)

Why ZeRO-3? Full fine-tuning = all 124M params need gradients + optimizer states.
ZeRO-3 shards model weights + gradients + optimizer across GPUs + CPU.

In [ ]:
!pip install -q transformers datasets accelerate deepspeed

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

In [ ]:
# Accelerate config — ZeRO-3 with CPU offload for full fine-tuning
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: true
  zero_stage: 3
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Config written. {NUM_GPUS} GPU(s), ZeRO-3, bf16, CPU offload.")

In [ ]:
%%writefile train_full.py
"""Full fine-tune GPT-2 on text data with DeepSpeed ZeRO-3."""
import torch, os, json, time
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, TrainerCallback,
    DataCollatorForLanguageModeling,
)

MODEL = "gpt2"

# Load tokenizer + model (full parameters, no LoRA)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL)
total_params = sum(p.numel() for p in model.parameters())
print(f"Full model: {total_params/1e6:.0f}M parameters (ALL trainable)")

# Load dataset — Wikitext is freely available, no auth needed
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
# Filter empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 50)
print(f"Dataset: {len(dataset)} text passages")

# Tokenize — pack into chunks of 512 tokens
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Tokenized: {len(dataset)} examples")

# Metrics callback
class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.logs = []
        self.train_start = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        torch.cuda.reset_peak_memory_stats()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.logs.append({"step": state.global_step, "loss": logs["loss"],
                             "lr": logs.get("learning_rate", 0)})

    def on_train_end(self, args, state, control, **kwargs):
        if int(os.environ.get("LOCAL_RANK", 0)) != 0:
            return
        train_time = time.time() - self.train_start
        num_gpus = int(os.environ.get("WORLD_SIZE", 1))
        metrics = {
            "model": MODEL,
            "method": "Full Fine-Tune (ZeRO-3)",
            "total_params": total_params,
            "trainable_params": total_params,
            "trainable_pct": 100.0,
            "num_gpus": num_gpus,
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_mem_allocated_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "gpu_mem_reserved_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2),
            "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
            "train_time_sec": round(train_time, 1),
            "total_steps": state.global_step,
            "final_loss": self.logs[-1]["loss"] if self.logs else None,
            "loss_history": self.logs,
        }
        with open("training_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"Metrics saved. {train_time:.0f}s, {state.global_step} steps")

# Train — full fine-tune, all parameters updated
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./gpt2_full_output",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        warmup_steps=50,
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[MetricsCallback()],
)

trainer.train()
trainer.save_model("./gpt2_full_output/final")
tokenizer.save_pretrained("./gpt2_full_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\nFull fine-tune complete!")

In [ ]:
print(f"Training GPT-2 (full) on {NUM_GPUS} GPU(s) with ZeRO-3...")
start = time.time()
!accelerate launch --num_processes={NUM_GPUS} train_full.py
elapsed = time.time() - start
print(f"\nDone! {elapsed:.0f}s on {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import pipeline

# Load fine-tuned model and generate
pipe = pipeline("text-generation", model="./gpt2_full_output/final", device="cuda")
prompts = ["The future of AI", "Once upon a time", "In recent years, scientists"]
for p in prompts:
    out = pipe(p, max_new_tokens=80, do_sample=True, temperature=0.8)[0]["generated_text"]
    print(f"Prompt: {p}")
    print(f"Output: {out}\n")

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as ticker
from IPython.display import HTML, display

with open("training_metrics.json") as f:
    m = json.load(f)

# --- Loss Curve + Learning Rate ---
fig, ax1 = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#0d1117")
ax1.set_facecolor("#0d1117")

steps = [h["step"] for h in m["loss_history"]]
losses = [h["loss"] for h in m["loss_history"]]
lrs = [h["lr"] for h in m["loss_history"]]

ax1.plot(steps, losses, color="#58a6ff", linewidth=2, label="Loss")
ax1.fill_between(steps, losses, alpha=0.1, color="#58a6ff")
ax1.set_xlabel("Step", color="#8b949e")
ax1.set_ylabel("Loss", color="#58a6ff")
ax1.tick_params(axis="y", labelcolor="#58a6ff")
ax1.tick_params(axis="x", colors="#8b949e")
ax1.grid(True, alpha=0.15, color="#30363d")
for s in ax1.spines.values(): s.set_color("#30363d")
ax1.spines["top"].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(steps, lrs, color="#f0883e", linewidth=1.5, linestyle="--", alpha=0.7, label="LR")
ax2.set_ylabel("Learning Rate", color="#f0883e")
ax2.tick_params(axis="y", labelcolor="#f0883e")
for s in ax2.spines.values(): s.set_color("#30363d")
ax2.spines["top"].set_visible(False)
ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1e"))

fig.suptitle("Training Loss & Learning Rate", color="#e6edf3", fontsize=14, fontweight="bold")
fig.legend(loc="upper right", bbox_to_anchor=(0.92, 0.88), facecolor="#161b22",
           edgecolor="#30363d", labelcolor="#e6edf3")
plt.tight_layout()
plt.show()

# --- GPU Memory Bar ---
fig2, ax3 = plt.subplots(figsize=(6, 3))
fig2.patch.set_facecolor("#0d1117")
ax3.set_facecolor("#0d1117")
labels = ["Allocated", "Reserved", "Total"]
vals = [m["gpu_mem_allocated_gb"], m["gpu_mem_reserved_gb"], m["gpu_mem_total_gb"]]
colors = ["#3fb950", "#58a6ff", "#30363d"]
bars = ax3.barh(labels, vals, color=colors, height=0.5)
for bar, val in zip(bars, vals):
    ax3.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f} GB",
             va="center", color="#e6edf3", fontsize=11, fontweight="bold")
ax3.set_xlim(0, m["gpu_mem_total_gb"] * 1.3)
ax3.set_title(f"GPU Memory - {m['gpu_name']}", color="#e6edf3", fontsize=13, fontweight="bold")
ax3.tick_params(colors="#8b949e")
for s in ax3.spines.values(): s.set_color("#30363d")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# --- HTML Dashboard ---
mem_util = m["gpu_mem_allocated_gb"] / m["gpu_mem_total_gb"] * 100
loss_drop = f"{(losses[0] - losses[-1]) / losses[0] * 100:.0f}% drop" if len(losses) >= 2 else ""

html = f"""
<div style="font-family: -apple-system, sans-serif; max-width: 780px; margin: 20px 0;">
  <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase;">Model</div>
      <div style="color: #58a6ff; font-size: 22px; font-weight: 700; margin: 6px 0;">GPT-2</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_params']/1e6:.0f}M params (100%)</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase;">Final Loss</div>
      <div style="color: #3fb950; font-size: 22px; font-weight: 700; margin: 6px 0;">{m['final_loss']:.3f}</div>
      <div style="color: #8b949e; font-size: 12px;">{loss_drop}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase;">Train Time</div>
      <div style="color: #f0883e; font-size: 22px; font-weight: 700; margin: 6px 0;">{m['train_time_sec']:.0f}s</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_steps']} steps</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase;">GPUs</div>
      <div style="color: #d2a8ff; font-size: 22px; font-weight: 700; margin: 6px 0;">{m['num_gpus']}x</div>
      <div style="color: #8b949e; font-size: 12px;">{m['gpu_name']}</div>
    </div>
  </div>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 10px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">Training Config</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 5px;">
        <tr><td style="color:#8b949e;">Method</td><td style="text-align:right; font-weight:600; color:#3fb950;">Full Fine-Tune</td></tr>
        <tr><td style="color:#8b949e;">Trainable</td><td style="text-align:right;">{m['trainable_params']/1e6:.0f}M / {m['total_params']/1e6:.0f}M (100%)</td></tr>
        <tr><td style="color:#8b949e;">DeepSpeed</td><td style="text-align:right;">ZeRO-3 + CPU offload</td></tr>
        <tr><td style="color:#8b949e;">Precision</td><td style="text-align:right;">bf16</td></tr>
      </table>
    </div>
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 10px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">GPU Memory</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 5px;">
        <tr><td style="color:#8b949e;">VRAM total</td><td style="text-align:right;">{m['gpu_mem_total_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Peak allocated</td><td style="text-align:right; color:#3fb950;">{m['gpu_mem_allocated_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Peak reserved</td><td style="text-align:right;">{m['gpu_mem_reserved_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Utilization</td><td style="text-align:right; color:#3fb950; font-weight:700;">{mem_util:.0f}%</td></tr>
      </table>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 14px 18px;
              margin-top: 12px; display: flex; justify-content: space-between; align-items: center;">
    <span style="color: #8b949e; font-size: 12px;">GPU Memory Utilization</span>
    <div style="flex: 1; margin: 0 16px; background: #21262d; border-radius: 6px; height: 18px; overflow: hidden;">
      <div style="width: {mem_util:.0f}%; height: 100%; border-radius: 6px;
                  background: linear-gradient(90deg, #238636, #3fb950);"></div>
    </div>
    <span style="color: #3fb950; font-size: 13px; font-weight: 700;">{mem_util:.0f}%</span>
  </div>
</div>
"""
display(HTML(html))

## Full Fine-Tune vs LoRA

| Aspect | LoRA | Full Fine-Tune |
|--------|------|---------------|
| Trainable params | ~1% | 100% |
| Memory needed | Low | High |
| DeepSpeed stage | ZeRO-2 sufficient | ZeRO-3 needed |
| Training speed | Fast | Slower |
| Quality ceiling | Good | Best possible |
| Use case | Quick adaptation | Maximum performance |

### How ZeRO-3 enables full fine-tuning on free GPUs

```
Without ZeRO-3 (won't fit on 2x T4):
  GPU 0: Full model (124M) + Full gradients + Full optimizer = ~2GB + 0.5GB + 2GB = 4.5GB
  GPU 1: Full model (124M) + Full gradients + Full optimizer = same

With ZeRO-3 (fits easily):
  GPU 0: 1/2 model + 1/2 gradients + 1/2 optimizer + CPU offload
  GPU 1: 1/2 model + 1/2 gradients + 1/2 optimizer + CPU offload
  -> Each GPU uses ~1.5GB instead of 4.5GB
```